In [27]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

DATA = Path("../dataset")
assert DATA.is_dir()
PLAN_COLS = ["tt_action_item_id", "tr_id", "time_begin", "geom"]
TRAFFIC_COLS = ["packet_id", "tr_id", "unit_id", "event_time", "gps_time",
                "speed", "receive_time", "is_hist_data", "location_valid", "lon", "lat"]

def read_split(split):
    points_path = DATA / ("validate/points.csv" if split == "validate"
                          else f"labels/labels_{split}.csv")
    points = pd.read_csv(points_path, dtype={"sample_id": str, "tr_id": str,
                                              "target_stop_id": str})
    plan_path = DATA / ("validate/schedule_plan.csv" if split == "validate"
                        else f"{split}/schedule.csv")
    # Фактические времена не читаем.
    plan = pd.read_csv(plan_path, usecols=PLAN_COLS,
                       dtype={"tr_id": str, "tt_action_item_id": str})
    traffic = pd.read_csv(DATA / split / "traffic.csv", usecols=TRAFFIC_COLS,
                          dtype={"packet_id": str, "tr_id": str, "unit_id": str},
                          low_memory=False)
    traffic = traffic.loc[traffic["tr_id"].isin(points["tr_id"])].copy()
    return points.reset_index(drop=True), plan, traffic

train, plan_train, traffic_train = read_split("train")
test, plan_test, traffic_test = read_split("test")
validate, plan_validate, traffic_validate = read_split("validate")
print("Samples:", len(train), len(test), len(validate))

Samples: 4434 353 151


In [28]:
STOP_SPEED_KMH = 2.0
WINDOWS = {"30s": 30, "1m": 60, "3m": 180, "5m": 300, "10m": 600, "15m": 900}

def haversine_m(lon1, lat1, lon2, lat2):
    r = 6_371_008.8
    a1, b1, a2, b2 = map(np.radians, [lon1, lat1, lon2, lat2])
    a = np.sin((b2-b1)/2)**2 + np.cos(b1)*np.cos(b2)*np.sin((a2-a1)/2)**2
    return 2*r*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def telemetry_at(history, now):
    out = {"unit_id": "missing", "packet_count_15m": 0, "packet_frequency": 0.0}
    if history.empty:
        return out
    last = history.iloc[-1]
    recent = history.loc[history["event_time"] > now - pd.Timedelta(minutes=15)]
    known_speed = history.dropna(subset=["speed"])
    out["unit_id"] = str(last["unit_id"]) if pd.notna(last["unit_id"]) else "missing"
    for col in ("event_time", "receive_time", "gps_time"):
        value = last[col]
        out[col] = value.value / 1e9 if pd.notna(value) else np.nan
    out["speed_now"] = known_speed["speed"].iloc[-1] if len(known_speed) else np.nan
    out["telemetry_age"] = (now - last["event_time"]).total_seconds()
    known_gps = history["gps_time"].dropna()
    out["gps_age"] = (now - known_gps.iloc[-1]).total_seconds() if len(known_gps) else np.nan
    out["receive_delay"] = (last["receive_time"] - last["event_time"]).total_seconds()
    out["last_packet_is_hist"] = float(last["is_hist_data"]) if pd.notna(last["is_hist_data"]) else np.nan
    out["packet_count_15m"] = len(recent)
    out["packet_frequency"] = len(recent) / 15
    out["hist_packet_fraction"] = recent["is_hist_data"].mean() if len(recent) else np.nan
    out["mean_receive_delay"] = ((recent["receive_time"] - recent["event_time"])
                                 .dt.total_seconds().mean() if len(recent) else np.nan)
    out["max_packet_gap"] = recent["event_time"].diff().dt.total_seconds().max()
    for suffix, seconds in WINDOWS.items():
        window = recent.loc[recent["event_time"] > now - pd.Timedelta(seconds=seconds)]
        valid = window.dropna(subset=["speed"])
        speeds = valid["speed"]
        for stat in ("mean", "std", "min", "max", "median"):
            out[f"speed_{stat}_{suffix}"] = getattr(speeds, stat)() if len(speeds) else np.nan
        if suffix in ("1m", "3m", "5m"):
            out[f"stopped_fraction_{suffix}"] = (speeds <= STOP_SPEED_KMH).mean() if len(speeds) else np.nan
            if len(valid) >= 2:
                span = (valid["event_time"].iloc[-1] - valid["event_time"].iloc[0]).total_seconds() / 60
                out[f"speed_slope_{suffix}"] = ((speeds.iloc[-1] - speeds.iloc[0]) / span
                                                 if span > 0 else np.nan)
        if suffix == "5m":
            longest, start = 0.0, None
            for event_time, speed in zip(valid["event_time"], speeds):
                if speed <= STOP_SPEED_KMH:
                    if start is None:
                        start = event_time
                    longest = max(longest, (event_time - start).total_seconds())
                else:
                    start = None
            out["longest_stop_5m"] = longest if len(valid) else np.nan
        if suffix == "10m":
            stopped = speeds <= STOP_SPEED_KMH
            out["stop_count_10m"] = int((stopped & ~stopped.shift(fill_value=False)).sum()) if len(valid) else 0
    if len(known_speed):
        moving = known_speed.loc[known_speed["speed"] > STOP_SPEED_KMH]
        out["time_since_last_move"] = ((now - moving["event_time"].iloc[-1]).total_seconds()
                                       if len(moving) else np.nan)
        if known_speed["speed"].iloc[-1] <= STOP_SPEED_KMH:
            prior_move = np.flatnonzero(known_speed["speed"].to_numpy() > STOP_SPEED_KMH)
            start_idx = prior_move[-1] + 1 if len(prior_move) else 0
            out["current_stop_duration"] = (now - known_speed["event_time"].iloc[start_idx]).total_seconds()
        else:
            out["current_stop_duration"] = 0.0
    return out

def spatial_at(history, now, target_lon, target_lat):
    out = {"sp_has_position": 0}
    valid = history.loc[history["location_valid"].eq(True)
                        & history["lon"].between(-180, 180)
                        & history["lat"].between(-90, 90)
                        & ~(history["lon"].eq(0) & history["lat"].eq(0))]
    if valid.empty:
        return out
    valid = valid.drop_duplicates("event_time", keep="last")
    current = valid.iloc[-1]
    age = (now - current["event_time"]).total_seconds()
    distance = haversine_m(current["lon"], current["lat"], target_lon, target_lat)
    out.update(sp_has_position=1, sp_position_age_s=age, sp_distance_to_target_m=distance)
    for minutes in (1, 3):
        query = now - pd.Timedelta(minutes=minutes)
        previous = valid.loc[valid["event_time"] <= query]
        if previous.empty or age > 60:
            continue
        past = previous.iloc[-1]
        if (query - past["event_time"]).total_seconds() > 60:
            continue
        elapsed = (current["event_time"] - past["event_time"]).total_seconds()
        if elapsed <= 0:
            continue
        old_distance = haversine_m(past["lon"], past["lat"], target_lon, target_lat)
        approach = old_distance - distance
        out[f"sp_approach_{minutes}m_m"] = approach
        out[f"sp_approach_speed_{minutes}m_mps"] = approach / elapsed
    return out

def build_features(points, plan, traffic):
    samples = points[["sample_id", "tr_id", "T", "target_stop_id",
                      "target_time_begin", "cur_dev_s"]].copy()
    for col in ("T", "target_time_begin"):
        samples[col] = pd.to_datetime(samples[col], format="mixed")
    plan = plan.copy()
    plan["time_begin"] = pd.to_datetime(plan["time_begin"], format="mixed")
    coords = plan["geom"].str.extract(r"POINT\s*\(\s*([-\d.]+)\s+([-\d.]+)\s*\)")
    plan["target_lon"] = pd.to_numeric(coords[0])
    plan["target_lat"] = pd.to_numeric(coords[1])
    target = plan[["tr_id", "tt_action_item_id", "time_begin", "target_lon", "target_lat"]].rename(
        columns={"tt_action_item_id": "target_stop_id", "time_begin": "target_plan_time"})
    samples = samples.merge(target, on=["tr_id", "target_stop_id"], how="left",
                            validate="many_to_one", sort=False)
    assert samples["target_lon"].notna().all() and samples["target_lat"].notna().all()
    assert samples["target_time_begin"].eq(samples["target_plan_time"]).all()
    samples["hour"] = samples["T"].dt.hour
    samples["minute"] = samples["T"].dt.minute
    samples["minute_of_day"] = samples["hour"]*60 + samples["minute"]
    samples["day_of_week"] = samples["T"].dt.dayofweek
    samples["is_weekend"] = (samples["day_of_week"] >= 5).astype(int)
    samples["sin_time"] = np.sin(2*np.pi*samples["minute_of_day"]/1440)
    samples["cos_time"] = np.cos(2*np.pi*samples["minute_of_day"]/1440)
    samples["target_hour"] = samples["target_time_begin"].dt.hour
    samples["target_minute_of_day"] = (samples["target_hour"]*60
                                      + samples["target_time_begin"].dt.minute)
    samples["forecast_horizon_s"] = (samples["target_time_begin"] - samples["T"]).dt.total_seconds()
    samples["abs_cur_dev"] = samples["cur_dev_s"].abs()
    samples["is_late_now"] = (samples["cur_dev_s"] > 0).astype(int)
    samples["is_early_now"] = (samples["cur_dev_s"] < 0).astype(int)
    by_plan = {trip: group.sort_values("time_begin") for trip, group in plan.groupby("tr_id")}
    samples["stops_to_target"] = [
        by_plan[row["tr_id"]].loc[
            (by_plan[row["tr_id"]]["time_begin"] > row["T"])
            & (by_plan[row["tr_id"]]["time_begin"] <= row["target_plan_time"]),
            "tt_action_item_id"].nunique()
        for _, row in samples.iterrows()
    ]

    traffic = traffic.copy()
    for col in ("event_time", "receive_time", "gps_time"):
        traffic[col] = pd.to_datetime(traffic[col], format="mixed", errors="coerce")
    traffic = traffic.dropna(subset=["event_time", "receive_time"])
    traffic = traffic.drop_duplicates(["tr_id", "packet_id"])
    by_trip = {trip: group.sort_values(["event_time", "packet_id"]).reset_index(drop=True)
               for trip, group in traffic.groupby("tr_id")}
    rows = []
    for _, row in samples.iterrows():
        now = row["T"]
        trip = by_trip.get(row["tr_id"])
        available = (trip.loc[(trip["event_time"] <= now) & (trip["receive_time"] <= now)]
                     if trip is not None else traffic.iloc[0:0])
        assert (available["event_time"] <= now).all() and (available["receive_time"] <= now).all()
        one = telemetry_at(available, now)
        one.update(spatial_at(available, now, row["target_lon"], row["target_lat"]))
        rows.append(one)
    result = pd.concat([samples.reset_index(drop=True),
                        pd.DataFrame.from_records(rows)], axis=1)
    assert len(result) == len(points) and result["sample_id"].is_unique
    assert not {"target_delay_s", "target_class", "time_fact_begin"}.intersection(result.columns)
    return result

F_train = build_features(train, plan_train, traffic_train)
F_test = build_features(test, plan_test, traffic_test)
F_validate = build_features(validate, plan_validate, traffic_validate)
assert F_train["sample_id"].equals(train["sample_id"])
assert F_test["sample_id"].equals(test["sample_id"])
assert F_validate["sample_id"].equals(validate["sample_id"])
print("Feature frames:", F_train.shape, F_test.shape, F_validate.shape)


Feature frames: (4434, 84) (353, 84) (151, 84)


In [29]:
SCHEDULE = ["tr_id", "target_stop_id", "hour", "minute", "minute_of_day",
            "day_of_week", "is_weekend", "sin_time", "cos_time", "target_hour",
            "target_minute_of_day", "forecast_horizon_s", "cur_dev_s",
            "abs_cur_dev", "is_late_now", "is_early_now", "stops_to_target"]
SPATIAL = ["sp_has_position", "sp_position_age_s", "sp_distance_to_target_m",
           "sp_approach_1m_m", "sp_approach_speed_1m_mps",
           "sp_approach_3m_m", "sp_approach_speed_3m_mps"]
TELEMETRY_54 = [
    "unit_id", "event_time", "receive_time", "gps_time", "speed_now",
    "telemetry_age", "gps_age", "receive_delay", "last_packet_is_hist",
    "packet_count_15m", "packet_frequency", "hist_packet_fraction",
    "mean_receive_delay", "max_packet_gap",
] + [f"speed_{stat}_{window}" for window in WINDOWS
     for stat in ("mean", "std", "min", "max", "median")] + [
    "stopped_fraction_1m", "stopped_fraction_3m", "stopped_fraction_5m",
    "speed_slope_1m", "speed_slope_3m", "speed_slope_5m",
    "longest_stop_5m", "stop_count_10m",
    "time_since_last_move", "current_stop_duration",
]
FEATURE_COLS = SCHEDULE + SPATIAL + TELEMETRY_54

In [30]:
CATEGORICAL = ["tr_id", "target_stop_id", "unit_id"]
def model_input(frame):
    x = frame[FEATURE_COLS].copy()
    for col in CATEGORICAL:
        x[col] = x[col].fillna("missing").astype(str)
    return x


model = CatBoostRegressor(loss_function="MAE",
                          iterations=200,
                          depth=7,
                          learning_rate=0.05,
                          l2_leaf_reg=5,
                          random_seed=42,
                          verbose=False,
                          allow_writing_files=False)
model.fit(model_input(F_train),
          train["target_delay_s"].to_numpy() - train["cur_dev_s"].to_numpy(),
          cat_features=CATEGORICAL)
model.save_model("final_model.cbm")
test_prediction = test["cur_dev_s"].to_numpy() + model.predict(model_input(F_test))
test_mae = np.mean(np.abs(test["target_delay_s"].to_numpy() - test_prediction))
print(f"Local test MAE: {test_mae:.4f} seconds")
prediction = validate["cur_dev_s"].to_numpy() + model.predict(model_input(F_validate))
pred = pd.DataFrame({"sample_id": F_validate["sample_id"], "prediction": prediction})
submission = pd.read_csv(DATA / "sample_submission.csv", sep=";", dtype={"sample_id": str})[["sample_id"]]
submission = submission.merge(pred, on="sample_id", how="left", validate="one_to_one")
assert len(submission) == len(validate) and np.isfinite(submission["prediction"]).all()
submission.to_csv("submission.csv", sep=";", index=False)

Local test MAE: 63.8795 seconds
